In [1]:
##SPRINT 3

In [2]:
## MILESTONE 5 AND 6

In [3]:
suppressPackageStartupMessages({
    library(shiny)
    library(shinydashboard)
    library(ggplot2)
    library(dplyr)
    library(plotly) 
    library(DT)     
    library(bsicons)
    library(shinycssloaders)
    library(countrycode)
    library(tidyverse)
    library(randomForest)
    library(xgboost)
    library(caret)
    library(pROC)
    library(ROSE)
})

In [4]:
set.seed(123)

In [5]:
df <- read.csv("CH4_N2O_Emissions_clean.csv")

In [6]:
df$ISO_Code <- countrycode(
  df$Country, 
  origin = 'country.name', 
  destination = 'iso3c',
  custom_match = c(
    "Micronesia" = "FSM",
    "St. Vin. and Gren." = "VCT"
  )
)

In [7]:
cat("Data loaded successfully!\n")
cat("Rows:", nrow(df), "| Columns:", ncol(df), "\n\n")

#Creating a target variable (1 = Dangerous, 0 = Safe)
df$is_dangerous <- ifelse(df$Overall_Safety == "Dangerous", 1, 0)

#Selecting the features
ml_data <- df %>%
  select(
    CH4.emissions.per.capita,
    N2O.emissions.per.capita,
    CH4.change.since.1990,
    N2O.change.since.1990,
    Total_Emissions,
    Emission_Ratio,
    Region,
    is_dangerous
  ) %>%
  drop_na()

cat("After cleaning missing values:\n")
cat("Rows:", nrow(ml_data), "rows\n\n")

cat("Class distribution:\n")
print(table(ml_data$is_dangerous))
cat("\n")

#CREATING A ONE-HOT ENCODING FOR REGION
region_dummies <- model.matrix(~ Region - 1, data = ml_data) %>% as.data.frame()
names(region_dummies) <- gsub("Region", "", names(region_dummies))

features <- ml_data %>%
  select(-Region, -is_dangerous) %>%
  bind_cols(region_dummies)

target <- ml_data$is_dangerous

cat("Number of features:", ncol(features), "\n\n")

#SPLITTING THE DATA

set.seed(123)
train_index <- createDataPartition(target, p = 0.8, list = FALSE)

X_train <- features[train_index, ]
X_test <- features[-train_index, ]
y_train <- target[train_index]
y_test <- target[-train_index]

cat("Training set:", nrow(X_train), "samples\n")
cat("Test set:", nrow(X_test), "samples\n\n")

#HANDLING THE CLASS IMBALANCE WITH ROSE

cat("Class balance before ROSE:\n")
print(table(y_train))
cat("\n")

set.seed(123)
balanced_data <- ROSE(is_dangerous ~ ., 
                      data = cbind(X_train, is_dangerous = y_train),
                      seed = 123)$data

X_train_balanced <- balanced_data %>% select(-is_dangerous)
y_train_balanced <- balanced_data$is_dangerous

cat("Class balance AFTER ROSE:\n")
print(table(y_train_balanced))
cat("\n")

#TRAINING USING THE RANDOM FOREST MODEL

cat("\n==================================================\n")
cat("TRAINING RANDOM FOREST MODEL...\n")
cat("==================================================\n\n")

set.seed(123)

#Converting to factor for classification
y_train_factor <- as.factor(y_train_balanced)

rf_model <- randomForest(
  x = X_train_balanced,
  y = y_train_factor,
  ntree = 500,
  mtry = round(sqrt(ncol(X_train_balanced))),
  importance = TRUE,
  keep.forest = TRUE
)

cat("Random Forest model trained!\n")
print(rf_model)

#Feature importance
importance_df <- as.data.frame(importance(rf_model))
importance_df$feature <- rownames(importance_df)
importance_df <- importance_df %>%
  arrange(desc(MeanDecreaseGini))

cat("\nTop 5 most important features:\n")
print(head(importance_df, 5))

#THE RANDOM FOREST PREDICTIONS
#Getting the prediction probabilities
rf_pred_probs_raw <- predict(rf_model, X_test, type = "prob")

#Checking for column names to find the positive class
cat("\nColumn names in prediction output:", colnames(rf_pred_probs_raw), "\n")

#The positive class (Dangerous = 1) is typically the second column or named "1"
if("1" %in% colnames(rf_pred_probs_raw)) {
  rf_pred_probs <- rf_pred_probs_raw[, "1"]
} else {
  rf_pred_probs <- rf_pred_probs_raw[, 2] 
}

rf_pred_class <- ifelse(rf_pred_probs > 0.5, 1, 0)

#Evaluating the Random Forest Model
rf_cm <- confusionMatrix(as.factor(rf_pred_class), as.factor(y_test))
rf_roc <- roc(y_test, rf_pred_probs)

cat("\n==================================================\n")
cat("RANDOM FOREST PERFORMANCE:\n")
cat("==================================================\n")
cat("Accuracy:", round(rf_cm$overall["Accuracy"], 4), "\n")
cat("Precision:", round(rf_cm$byClass["Precision"], 4), "\n")
cat("Recall:", round(rf_cm$byClass["Sensitivity"], 4), "\n")

rf_f1 <- 2 * (rf_cm$byClass["Precision"] * rf_cm$byClass["Sensitivity"]) / 
         (rf_cm$byClass["Precision"] + rf_cm$byClass["Sensitivity"])
cat("F1 Score:", round(rf_f1, 4), "\n")
cat("AUC:", round(auc(rf_roc), 4), "\n")

#TRAINING THE XGBOOST MODEL

cat("\n==================================================\n")
cat("TRAINING XGBOOST MODEL...\n")
cat("==================================================\n\n")

X_train_matrix <- as.matrix(X_train_balanced)
X_test_matrix <- as.matrix(X_test)

dtrain <- xgb.DMatrix(X_train_matrix, label = y_train_balanced)
dtest <- xgb.DMatrix(X_test_matrix, label = y_test)

params <- list(
  objective = "binary:logistic",
  max_depth = 4,
  eta = 0.05,
  eval_metric = "auc",
  subsample = 0.8,
  colsample_bytree = 0.8,
  min_child_weight = 1
)

set.seed(123)
xgb_cv <- xgb.cv(
  params = params,
  data = dtrain,
  nrounds = 100,
  nfold = 5,
  early_stopping_rounds = 10,
  verbose = 0
)

best_nrounds <- which.max(xgb_cv$evaluation_log$test_auc_mean)
cat("Optimal number of rounds:", best_nrounds, "\n")

xgb_model <- xgb.train(
  params = params,
  data = dtrain,
  nrounds = best_nrounds,
  verbose = 0
)

cat("XGBoost model trained!\n")

#XGBOOST PREDICTIONS

xgb_pred_probs <- predict(xgb_model, dtest)
xgb_pred_class <- ifelse(xgb_pred_probs > 0.5, 1, 0)

xgb_cm <- confusionMatrix(as.factor(xgb_pred_class), as.factor(y_test))
xgb_roc <- roc(y_test, xgb_pred_probs)

cat("\n==================================================\n")
cat("XGBOOST PERFORMANCE:\n")
cat("==================================================\n")
cat("Accuracy:", round(xgb_cm$overall["Accuracy"], 4), "\n")
cat("Precision:", round(xgb_cm$byClass["Precision"], 4), "\n")
cat("Recall:", round(xgb_cm$byClass["Sensitivity"], 4), "\n")

xgb_f1 <- 2 * (xgb_cm$byClass["Precision"] * xgb_cm$byClass["Sensitivity"]) / 
          (xgb_cm$byClass["Precision"] + xgb_cm$byClass["Sensitivity"])
cat("F1 Score:", round(xgb_f1, 4), "\n")
cat("AUC:", round(auc(xgb_roc), 4), "\n")

#COMPARING THE MODELS

cat("\n==================================================\n")
cat("MODEL COMPARISON:\n")
cat("==================================================\n")

comparison <- data.frame(
  Metric = c("Accuracy", "Precision", "Recall", "F1 Score", "AUC"),
  RandomForest = c(
    round(rf_cm$overall["Accuracy"], 4),
    round(rf_cm$byClass["Precision"], 4),
    round(rf_cm$byClass["Sensitivity"], 4),
    round(rf_f1, 4),
    round(auc(rf_roc), 4)
  ),
  XGBoost = c(
    round(xgb_cm$overall["Accuracy"], 4),
    round(xgb_cm$byClass["Precision"], 4),
    round(xgb_cm$byClass["Sensitivity"], 4),
    round(xgb_f1, 4),
    round(auc(xgb_roc), 4)
  )
)

print(comparison)

#SELECTING AND SAVING THE BEST MODEL

rf_auc <- auc(rf_roc)
xgb_auc <- auc(xgb_roc)

if(rf_auc >= xgb_auc) {
  best_model <- rf_model
  best_model_name <- "RandomForest"
  cat("\n Selected Random Forest as best model (AUC:", round(rf_auc, 4), ")\n")
} else {
  best_model <- xgb_model
  best_model_name <- "XGBoost"
  cat("\n Selected XGBoost as best model (AUC:", round(xgb_auc, 4), ")\n")
}

saveRDS(best_model, "best_emission_model.rds")
saveRDS(features, "model_features.rds")

cat("\n Model saved as 'best_emission_model.rds'\n")
cat(" Feature template saved as 'model_features.rds'\n")

#CREATING VISUALIZATIONS

#Feature importance plot
importance_plot <- ggplot(head(importance_df, 10), 
                          aes(x = reorder(feature, MeanDecreaseGini), 
                              y = MeanDecreaseGini)) +
  geom_bar(stat = "identity", fill = "#2E86C1") +
  coord_flip() +
  theme_minimal() +
  labs(
    title = "Top 10 Features Predicting Dangerous Emissions",
    x = "Feature",
    y = "Importance (Mean Decrease Gini)"
  )

ggsave("feature_importance.png", importance_plot, width = 8, height = 6)
cat(" Feature importance chart saved as 'feature_importance.png'\n")

#ROC plot
rf_roc_data <- data.frame(
  fpr = 1 - rf_roc$specificities,
  tpr = rf_roc$sensitivities,
  model = "Random Forest"
)

xgb_roc_data <- data.frame(
  fpr = 1 - xgb_roc$specificities,
  tpr = xgb_roc$sensitivities,
  model = "XGBoost"
)

roc_combined <- rbind(rf_roc_data, xgb_roc_data)

roc_plot <- ggplot(roc_combined, aes(x = fpr, y = tpr, color = model)) +
  geom_line(size = 1.2) +
  geom_abline(linetype = "dashed", alpha = 0.5) +
  theme_minimal() +
  labs(
    title = "ROC Curves: Random Forest vs XGBoost",
    x = "False Positive Rate",
    y = "True Positive Rate",
    color = "Model"
  ) +
  scale_color_manual(values = c("Random Forest" = "#2E86C1", "XGBoost" = "#E67E22"))

ggsave("roc_comparison.png", roc_plot, width = 7, height = 5)
cat(" ROC comparison chart saved as 'roc_comparison.png'\n")


#CREATING THE COMPLETION MESSAGE

cat("\n==================================================\n")
cat("STEP 1 COMPLETE!\n")
cat("==================================================\n")
cat("\nFiles created:\n")
cat("  - best_emission_model.rds\n")
cat("  - model_features.rds\n")
cat("  - feature_importance.png\n")
cat("  - roc_comparison.png\n")

Data loaded successfully!
Rows: 183 | Columns: 17 

After cleaning missing values:
Rows: 183 rows

Class distribution:

  0   1 
155  28 

Number of features: 11 

Training set: 147 samples
Test set: 36 samples

Class balance before ROSE:
y_train
  0   1 
126  21 

Class balance AFTER ROSE:
y_train_balanced
 0  1 
76 71 


TRAINING RANDOM FOREST MODEL...

Random Forest model trained!

Call:
 randomForest(x = X_train_balanced, y = y_train_factor, ntree = 500,      mtry = round(sqrt(ncol(X_train_balanced))), importance = TRUE,      keep.forest = TRUE) 
               Type of random forest: classification
                     Number of trees: 500
No. of variables tried at each split: 3

        OOB estimate of  error rate: 5.44%
Confusion matrix:
   0  1 class.error
0 73  3  0.03947368
1  5 66  0.07042254

Top 5 most important features:
                                 0         1 MeanDecreaseAccuracy
Total_Emissions          30.664288 23.415678            30.377852
N2O.emissions.per.capi

Setting levels: control = 0, case = 1

Setting direction: controls < cases




RANDOM FOREST PERFORMANCE:
Accuracy: 0.9444 
Precision: 0.9355 
Recall: 1 
F1 Score: 0.9667 
AUC: 0.9655 

TRAINING XGBOOST MODEL...

Optimal number of rounds: 4 
XGBoost model trained!


Setting levels: control = 0, case = 1

Setting direction: controls < cases




XGBOOST PERFORMANCE:
Accuracy: 0.8333 
Precision: 0.8485 
Recall: 0.9655 
F1 Score: 0.9032 
AUC: 0.7734 

MODEL COMPARISON:
     Metric RandomForest XGBoost
1  Accuracy       0.9444  0.8333
2 Precision       0.9355  0.8485
3    Recall       1.0000  0.9655
4  F1 Score       0.9667  0.9032
5       AUC       0.9655  0.7734

 Selected Random Forest as best model (AUC: 0.9655 )

 Model saved as 'best_emission_model.rds'
 Feature template saved as 'model_features.rds'
 Feature importance chart saved as 'feature_importance.png'


Warning message:
"Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead."


 ROC comparison chart saved as 'roc_comparison.png'

STEP 1 COMPLETE!

Files created:
  - best_emission_model.rds
  - model_features.rds
  - feature_importance.png
  - roc_comparison.png


In [8]:
#Loading Machine Learning models
best_model <- readRDS("best_emission_model.rds")
model_features <- readRDS("model_features.rds")

In [9]:
#Checking for model type
model_type <- ifelse(class(best_model)[1] == "randomForest", "randomForest", "xgb.Booster")
cat("Loaded model type:", model_type, "\n")
cat("Number of features expected:", ncol(model_features), "\n")

Loaded model type: randomForest 
Number of features expected: 11 


In [10]:
#Extracting the region levels from the model features
region_columns <- grep("^[A-Z]", names(model_features), value = TRUE)
region_columns <- region_columns[!region_columns %in% c("CH4.emissions.per.capita", 
                                                         "N2O.emissions.per.capita",
                                                         "CH4.change.since.1990",
                                                         "N2O.change.since.1990",
                                                         "Total_Emissions",
                                                         "Emission_Ratio")]
cat("Region columns in model:", paste(region_columns, collapse = ", "), "\n")

Region columns in model: Africa, Americas, Asia, Europe, Oceania 


In [ ]:

ui <- dashboardPage(
  title = "GROUP 12 Analytical System",
  skin = "green",
  
  dashboardHeader(
    title = tagList(
      bs_icon("globe-central-south-asia"), 
      " Emissions System"
    ),
    titleWidth = 300
  ),
  
  dashboardSidebar(
    sidebarMenu(
      menuItem("📊 Dashboard", tabName = "dashboard", icon = icon("dashboard")),
      menuItem("🗺️ World Map", tabName = "map", icon = icon("globe")),
      menuItem("📈 Statistical Models", tabName = "stats", icon = icon("chart-line")),
      menuItem("📋 Data Explorer", tabName = "data", icon = icon("table")),
      menuItem("🤖 AI Predictor", tabName = "predictor", icon = icon("robot")),
      menuItem("🎯 Intervention Planner", tabName = "intervention", icon = icon("chart-simple"))
    ),
    
    hr(),
    h4("🔍 Global Filters", style = "padding-left: 15px;"),
    
    selectInput("region_filter", "Select Region:", 
                choices = c("All", sort(unique(df$Region))), selected = "All"),
    
    #sliderInput
    
    br(),
    div(style = "padding: 15px;",
        helpText("Data Source: Group 12 Research Output"),
        helpText(paste("🤖 Model:", ifelse(model_type == "randomForest", "Random Forest", "XGBoost")))
    )
  ),
  
  dashboardBody(
    tags$head(
      tags$link(rel = "shortcut icon", type = "image/x-icon", href = "favicon.ico"),
      tags$style(HTML("
        .small-box { min-height: 120px !important; }
        
        .small-box p { 
          white-space: nowrap; 
          font-size: 13px !important; 
        }
        
        .small-box .icon-large { font-size: 50px !important; }

        .content-wrapper, .right-side { background-color: #f4f7f6; }
        .main-header .logo { 
          background-color: #288C41 !important; 
          font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
          font-weight: bold;
        }
        .main-header .navbar { background-color: #288C41 !important; }
      "))
    ),
    
    tabItems(
      #THE DASHBOARD PAGE
      tabItem(tabName = "dashboard",
        fluidRow(
          valueBoxOutput("total_countries", width = 4),
          valueBoxOutput("avg_ch4", width = 2),
          valueBoxOutput("avg_n2o", width = 2),
          valueBoxOutput("dangerous_pct", width = 4)
        ),
        fluidRow(
          box(title = "Regional Emission Comparison", status = "success", solidHeader = TRUE, width = 8,
              plotlyOutput("region_bar")),
          box(title = "Pollution Profile", status = "warning", solidHeader = TRUE, width = 4,
              plotlyOutput("profile_pie"))
        )
      ),
      
      #THE WORLD MAP PAGE
      tabItem(tabName = "map",
        fluidRow(
          box(
            title = "Global Emission Safety Choropleth",
            status = "primary", 
            solidHeader = TRUE, 
            width = 12,
            plotlyOutput("world_map", height = "600px") %>% withSpinner()
          )
        ),
        fluidRow(
          box(title = "Decision Support Note", width = 12,
            p(strong("How to use this map:"),
              tags$ul(
                tags$li("🟦 Blue countries = Safe emission levels"),
                tags$li("🟥 Red countries = Dangerous emission levels"),
                tags$li("👆 Hover over any country to see detailed statistics")
              ))
          )
        )
      ),
      
      #THE STATISTICS PAGE
      tabItem(tabName = "stats",
        fluidRow(
          box(title = "Regression Analysis", status = "info", solidHeader = TRUE, width = 6,
              verbatimTextOutput("reg_summary")),
          box(title = "Scatter Plot", status = "info", solidHeader = TRUE, width = 6,
              plotlyOutput("scatter_plot"))
        )
      ),
      
      #THE DATA EXPLORER PAGE
      tabItem(tabName = "data",
        box(title = "Emissions Dataset", width = 12, DTOutput("data_table"))
      ),

      #THE AI PREDICTOR TAB
      tabItem(tabName = "predictor",
        fluidRow(
          box(title = "Country Risk Predictor", status = "success", solidHeader = TRUE, width = 6,
            selectInput("predict_country", "Select a Country:", 
                        choices = sort(unique(df$Country))),
            br(),
            h4("Prediction Result:"),
            verbatimTextOutput("prediction_result"),
            br(),
            h4("Risk Factors:"),
            verbatimTextOutput("risk_factors")
          ),
          box(title = "What-If Simulation", status = "info", solidHeader = TRUE, width = 6,
            p("Adjust emissions to see how risk changes:"),
            sliderInput("sim_ch4", "CH₄ Emissions (Mt)", 
                        min = 0, max = 600, value = 50, step = 10),
            sliderInput("sim_n2o", "N₂O Emissions (Mt)", 
                        min = 0, max = 400, value = 20, step = 10),
            h4("Simulated Risk:"),
            verbatimTextOutput("simulation_result")
          )
        ),
        fluidRow(
          box(title = "How to Use", width = 12,
            p("Select a country to see its current risk prediction. Use the sliders to simulate 'what-if' scenarios.")
          )
        )
      ),

     #THE INTERVENTION PLANNER TAB
    tabItem(tabName = "intervention",
      fluidRow(
        box(title = "Priority Countries for Intervention (Based on Total Emissions)", 
            status = "danger", solidHeader = TRUE, width = 12,
          DTOutput("priority_table"),
          br(),
          downloadButton("download_priority", "Download Priority List (CSV)", class = "btn-success")
        )
      ),
      fluidRow(
        box(title = "How Priority Score is Calculated", width = 6,
          p("The Intervention Priority Score (IPS) combines three factors:"),
          tags$ul(
            tags$li("🔴 <strong>Risk Status (50%)</strong> - Currently Dangerous countries get highest weight"),
            tags$li("📊 <strong>Total Emission Volume (30%)</strong> - Total CH₄ + N₂O emissions"),
            tags$li("👥 <strong>Per-Capita Impact (20%)</strong> - Emissions per person (equity consideration)")
          ),
          p(em("IPS = (Risk × 0.5) + (Volume × 0.3) + (PerCapita × 0.2), normalized to 0-20 scale"))
        ),
        box(title = "Top Priority Explanation", width = 6,
          htmlOutput("priority_explanation")
        )
        )
      )
    )
  )
)

server <- function(input, output, session) {
  
  #Reactivating the Data Filtering
  filtered_df <- reactive({
    data <- df
    if(input$region_filter != "All") {
      data <- data %>% filter(Region == input$region_filter)
    }
    data
  })
  
  #Preparing map data
  map_data <- reactive({
    data <- filtered_df() %>%
      filter(!is.na(ISO_Code)) %>%
      mutate(
        Total_Emission = CH4.emissions + N2O.emissions,
        hover_text = paste0(
          "<b>", Country, "</b><br>",
          "────────────────<br>",
          "🏷️ <b>Safety Status:</b> ", Methane_Safety, "<br>",
          "────────────────<br>",
          "💨 <b>CH₄:</b> ", format(round(CH4.emissions, 2), big.mark = ","), " Mt<br>",
          "💨 <b>N₂O:</b> ", format(round(N2O.emissions, 2), big.mark = ","), " Mt<br>",
          "📊 <b>Total:</b> ", format(round(Total_Emission, 2), big.mark = ","), " Mt<br>",
          "────────────────<br>",
          "🌍 <b>Region:</b> ", Region
        ),
        safety_numeric = ifelse(Methane_Safety == "Safe", 1, 2)
      )
    data
  })

  prepare_features <- function(country_data) {
    country_data <- country_data %>%
      mutate(
        Total_Emissions = CH4.emissions + N2O.emissions,
        Emission_Ratio = CH4.emissions / (N2O.emissions + 0.001)
      )
    
    features <- country_data %>%
      select(
        CH4.emissions.per.capita,
        N2O.emissions.per.capita,
        CH4.change.since.1990,
        N2O.change.since.1990,
        Total_Emissions,
        Emission_Ratio
      )

    #Adding region dummy variables for one-hot encoding
    region_name <- country_data$Region[1]
    
    #Creating a dummy variables for ALL possible regions from model
    for(region_col in region_columns) {
      features[[region_col]] <- ifelse(region_name == region_col, 1, 0)
    }
    
    #Ensuring all expected columns exist in correct order
    for(col in names(model_features)) {
      if(!col %in% names(features)) {
        features[[col]] <- 0
      }
    }
    
    #Reordering to match model features exactly
    features <- features[, names(model_features), drop = FALSE]
    
    #Converting to matrix for XGBoost
    if(model_type == "xgb.Booster") {
      features <- as.matrix(features)
    }
    
    return(features)
  }

  #COUNTRY PREDICTION
  
  output$prediction_result <- renderPrint({
    req(input$predict_country)
    
    country_data <- df %>% filter(Country == input$predict_country)
    
    if(nrow(country_data) == 0) {
      cat("❌ No data available for this country.")
      return()
    }
    
    #Preparing features
    features <- tryCatch({
      prepare_features(country_data)
    }, error = function(e) {
      cat("Error preparing features:", e$message, "\n")
      return(NULL)
    })
    
    if(is.null(features)) {
      cat("❌ Error preparing features for prediction.")
      return()
    }
    
    #Making predictions based on model type
    prob_dangerous <- tryCatch({
      if(model_type == "randomForest") {
        #Random Forest prediction
        pred_prob_raw <- predict(best_model, features, type = "prob")
        
        #Handle different output formats
        if(is.data.frame(pred_prob_raw) && "1" %in% colnames(pred_prob_raw)) {
          pred_prob_raw[, "1"]
        } else if(is.matrix(pred_prob_raw) && ncol(pred_prob_raw) == 2) {
          pred_prob_raw[, 2]
        } else if(is.numeric(pred_prob_raw)) {
          pred_prob_raw[1]
        } else {
          0.5
        }
      } else {
        #XGBoost prediction
        pred <- predict(best_model, features)
        if(length(pred) > 1) pred[1] else pred
      }
    }, error = function(e) {
      cat("Prediction error:", e$message, "\n")
      return(0.5)
    })
    
    #Ensuring single numeric value
    prob_dangerous <- as.numeric(prob_dangerous)[1]
    if(is.na(prob_dangerous)) prob_dangerous <- 0.5
    
    actual_status <- country_data$Methane_Safety[1]
    
    #Displaying results
    cat("🌍 Country:", input$predict_country, "\n")
    cat("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")
    cat("📊 CURRENT STATUS:", actual_status, "\n")
    cat("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")
    cat("🤖 AI PREDICTION:\n")
    
    if(prob_dangerous > 0.5) {
      cat("   ⚠️ DANGEROUS (", round(prob_dangerous * 100, 1), "% confidence)\n")
      cat("   💡 Recommendation: Immediate policy intervention required\n")
    } else {
      cat("   ✅ SAFE (", round((1 - prob_dangerous) * 100, 1), "% confidence)\n")
      cat("   💡 Recommendation: Maintain current policies\n")
    }
    
    #Insights
    if(actual_status == "Dangerous" && prob_dangerous <= 0.5) {
      cat("\n🔍 INSIGHT: Model predicts improvement potential!\n")
      cat("   Current policies may already be reducing emissions.\n")
    } else if(actual_status == "Safe" && prob_dangerous > 0.5) {
      cat("\n⚠️ WARNING: Country is at risk of becoming Dangerous!\n")
      cat("   Precautionary measures recommended.\n")
    } else if(actual_status == "Dangerous" && prob_dangerous > 0.7) {
      cat("\n🚨 CRITICAL: High confidence in dangerous classification\n")
      cat("   Urgent intervention needed.\n")
    }
  })

  #THE RISK FACTORS EXPLANATION
  
  output$risk_factors <- renderPrint({
    req(input$predict_country)
    
    country_data <- df %>% filter(Country == input$predict_country)
    
    if(nrow(country_data) == 0) {
      cat("No data available")
      return()
    }
    
    total_emission <- country_data$CH4.emissions + country_data$N2O.emissions
    
    cat("📊 Key Risk Indicators for", input$predict_country, ":\n")
    cat("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")
    cat("💨 CH₄ per capita:", round(country_data$CH4.emissions.per.capita, 2), "Mt\n")
    cat("💨 N₂O per capita:", round(country_data$N2O.emissions.per.capita, 2), "Mt\n")
    cat("📈 CH₄ change (1990-2020):", round(country_data$CH4.change.since.1990, 2), "%\n")
    cat("📈 N₂O change (1990-2020):", round(country_data$N2O.change.since.1990, 2), "%\n")
    cat("⚖️ CH₄:N₂O Ratio:", round(country_data$Emission_Ratio, 2), "\n")
    cat("📊 Total Emissions:", round(total_emission, 2), "Mt\n")
    cat("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")
    
    #Trend analysis
    if(country_data$CH4.change.since.1990 > 10) {
      cat("⚠️ CRITICAL: Methane emissions increased >10% since 1990\n")
    } else if(country_data$CH4.change.since.1990 > 0) {
      cat("⚠️ Methane emissions are INCREASING since 1990\n")
    } else {
      cat("✅ Methane emissions are DECREASING since 1990\n")
    }
    
    #Profile analysis
    if(country_data$Emission_Ratio > 3) {
      cat("🔥 Methane-dominant pollution profile (high global warming impact)\n")
    } else if(country_data$Emission_Ratio < 0.5) {
      cat("🔥 Nitrous-oxide-dominant pollution profile (long-term threat)\n")
    } else {
      cat("📊 Balanced pollution profile\n")
    }
    
    if(total_emission > 200) {
      cat("🏭 MAJOR EMITTER: Top-tier emission levels\n")
    } else if(total_emission > 100) {
      cat("🏭 Significant contributor to global emissions\n")
    }
  })

  #SIMULATION OUTPUT
  
  output$simulation_result <- renderPrint({
    sim_ch4_val <- input$sim_ch4
    sim_n2o_val <- input$sim_n2o
    
    #Thresholds based on data analysis
    ch4_threshold <- 50
    n2o_threshold <- 18
    
    if(sim_ch4_val > ch4_threshold || sim_n2o_val > n2o_threshold) {
      cat("⚠️ SIMULATED RISK: DANGEROUS\n")
      cat("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")
      cat("With these emission levels:\n")
      cat("  - CH₄:", sim_ch4_val, "Mt (threshold:", ch4_threshold, "Mt)\n")
      cat("  - N₂O:", sim_n2o_val, "Mt (threshold:", n2o_threshold, "Mt)\n")
      
      ch4_reduction_needed <- max(0, sim_ch4_val - ch4_threshold)
      n2o_reduction_needed <- max(0, sim_n2o_val - n2o_threshold)
      
      cat("\n💡 INTERVENTION TARGETS:\n")
      if(ch4_reduction_needed > 0) {
        cat("   Reduce CH₄ by", ch4_reduction_needed, "Mt\n")
      }
      if(n2o_reduction_needed > 0) {
        cat("   Reduce N₂O by", n2o_reduction_needed, "Mt\n")
      }
      
      cat("\n📋 RECOMMENDED ACTIONS:\n")
      if(ch4_reduction_needed > 0) {
        cat("   • Improve methane capture in agriculture/waste\n")
        cat("   • Reduce livestock emissions\n")
        cat("   • Fix natural gas leaks\n")
      }
      if(n2o_reduction_needed > 0) {
        cat("   • Optimize fertilizer use\n")
        cat("   • Improve industrial processes\n")
      }
    } else {
      cat("✅ SIMULATED RISK: SAFE\n")
      cat("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")
      cat("These emission levels are within safe thresholds.\n")
      
      ch4_room <- ch4_threshold - sim_ch4_val
      n2o_room <- n2o_threshold - sim_n2o_val
      
      cat("\n📊 Safety buffer:\n")
      cat("   CH₄: can increase by", round(ch4_room, 1), "Mt before becoming dangerous\n")
      cat("   N₂O: can increase by", round(n2o_room, 1), "Mt before becoming dangerous\n")
    }
  })

  #KPI Boxes
  output$total_countries <- renderValueBox({
    valueBox(nrow(filtered_df()), "Countries Analyzed", 
             icon = icon("flag"), color = "green")
  })
  
  output$avg_ch4 <- renderValueBox({
    val <- round(mean(filtered_df()$CH4.emissions, na.rm = TRUE), 2)
    valueBox(val, "Average Methane (Mt)", icon = icon("cloud"), color = "blue")
  })
    
  output$avg_n2o <- renderValueBox({
    val <- round(mean(filtered_df()$N2O.emissions, na.rm = TRUE), 2)
    valueBox(val, "Average Nitrous Oxide (Mt)", icon = icon("cloud"), color = "orange")
  })
  

  output$dangerous_pct <- renderValueBox({
    pct <- round((sum(filtered_df()$Methane_Safety == "Dangerous", na.rm = TRUE) / 
                    nrow(filtered_df())) * 100, 1)
    valueBox(paste0(pct, "%"), "Dangerous Emitters", 
             icon = icon("warning"), color = "red")
  })

  # PRIORITY TABLE (REACTIVE)
  
 priority_data <- reactive({
  filtered_df() %>%
    filter(!is.na(Overall_Safety)) %>%
    mutate(
      #The priority score components based on TOTAL emissions
      risk_score = ifelse(Overall_Safety == "Dangerous", 10, 0),
      volume_score = Total_Emissions / max(Total_Emissions, na.rm = TRUE) * 5,
      per_capita_score = (CH4.emissions.per.capita + N2O.emissions.per.capita) / 
                         max(CH4.emissions.per.capita + N2O.emissions.per.capita, na.rm = TRUE) * 5,
      
      #Final priority score
      Priority_Score = round(risk_score + volume_score + per_capita_score, 2),
      
      #Intervention recommendation based on TOTAL emissions
      Recommendation = case_when(
        Overall_Safety == "Dangerous" & Priority_Score > 12 ~ "URGENT - Immediate action required",
        Overall_Safety == "Dangerous" & Priority_Score <= 12 ~ "High priority - Develop action plan",
        Overall_Safety == "Safe" & (CH4.change.since.1990 > 5 | N2O.change.since.1990 > 5) ~ "Monitor - Increasing trend detected",
        TRUE ~ "Low priority - Maintain current policies"
      )
    ) %>%
    arrange(desc(Priority_Score)) %>%
    select(Country, Region, Total_Emissions, CH4.emissions, N2O.emissions, 
           Overall_Safety, Priority_Score, Recommendation)
})
  output$priority_table <- renderDT({
  datatable(
    priority_data(),
    options = list(
      pageLength = 15,
      scrollX = TRUE,
      columnDefs = list(
        list(className = 'dt-center', targets = '_all')
      )
    ),
    rownames = FALSE
  ) %>%
    formatStyle("Overall_Safety",
      backgroundColor = styleEqual(c("Safe", "Dangerous"), c("#90EE90", "#FF6B6B"))
    ) %>%
    formatStyle("Priority_Score",
      background = styleColorBar(c(0, 20), "#FFA500"),
      backgroundSize = '100% 90%',
      backgroundRepeat = 'no-repeat',
      backgroundPosition = 'center'
    ) %>%
    formatRound(columns = c("Total_Emissions", "CH4.emissions", "N2O.emissions"), digits = 2)
})
  #Download handler
  output$download_priority <- downloadHandler(
  filename = function() {
    paste("intervention_priorities_total_emissions_", Sys.Date(), ".csv", sep = "")
  },
  content = function(file) {
    write.csv(priority_data(), file, row.names = FALSE)
  }
)

  #Priority explanation
output$priority_explanation <- renderUI({
  req(nrow(priority_data()) > 0)
  top_country <- priority_data()[1, ]
  
  HTML(paste0(
    "<div style='background-color: #f8f9fa; padding: 15px; border-radius: 8px;'>",
    "<h4>Why is ", top_country$Country, " the top priority?</h4>",
    "<p><strong>Region:</strong> ", top_country$Region, "</p>",
    "<p><strong>Overall Safety Status:</strong> ", top_country$Overall_Safety, "</p>",
    "<p><strong>Total Emissions (CH₄ + N₂O):</strong> ", round(top_country$Total_Emissions, 2), " Mt</p>",
    "<p><strong>CH₄ Emissions:</strong> ", round(top_country$CH4.emissions, 2), " Mt</p>",
    "<p><strong>N₂O Emissions:</strong> ", round(top_country$N2O.emissions, 2), " Mt</p>",
    "<p><strong>Priority Score:</strong> ", top_country$Priority_Score, "/20</p>",
    "<p><strong>Recommended Action:</strong> ", top_country$Recommendation, "</p>",
    "<hr>",
    "<p><em>Priority scores combine: current risk status (50%), total emission volume (30%), and per-capita impact (20%).</em></p>",
    "</div>"
  ))
})

  #WORLD MAP
  output$world_map <- renderPlotly({
    req(map_data())
    plot_data <- map_data()
    
    if(nrow(plot_data) == 0) {
      return(plot_ly() %>% 
               layout(title = "No data available for selected filters"))
    }
    
    plot_ly(
      type = 'choropleth',
      locations = plot_data$ISO_Code,
      z = plot_data$safety_numeric,
      text = plot_data$hover_text,
      hoverinfo = 'text',
      colorscale = list(
        list(0.0, 'skyblue'),
        list(1.0, 'red')
      ),
      zmin = 1,
      zmax = 2,
      colorbar = list(
        title = "Safety Status",
        tickvals = c(1, 2),
        ticktext = c("Safe", "Dangerous"),
        len = 0.3
      ),
      marker = list(line = list(color = 'rgb(200,200,200)', width = 0.5))
    ) %>%
      layout(
        title = "Global Emission Safety Assessment",
        geo = list(
          showframe = FALSE,
          showcoastlines = TRUE,
          projection = list(type = 'natural earth'),
          showland = TRUE,
          landcolor = 'rgb(240,240,240)'
        ),
        hoverlabel = list(
          bgcolor = "white",
          bordercolor = "black",
          font = list(size = 12)
        )
      ) %>%
      config(displayModeBar = TRUE, scrollZoom = TRUE)
  })

  #Regional Bar Chart
  output$region_bar <- renderPlotly({
    region_summary <- filtered_df() %>%
      group_by(Region) %>%
      summarise(
        Avg_CH4 = mean(CH4.emissions, na.rm = TRUE),
        Avg_N2O = mean(N2O.emissions, na.rm = TRUE),
        .groups = 'drop'
      )
    
    plot_ly(region_summary, x = ~Region, y = ~Avg_CH4, name = "CH₄",
            type = "bar", marker = list(color = "#2E86C1")) %>%
      add_trace(y = ~Avg_N2O, name = "N₂O", marker = list(color = "#E67E22")) %>%
      layout(barmode = "group", 
             title = "Average Emissions by Region",
             xaxis = list(title = "", tickangle = -45),
             yaxis = list(title = "Emissions (Mt)"))
  })
  
  #Pollution Profile Pie Chart
  output$profile_pie <- renderPlotly({
    safety_counts <- filtered_df() %>%
      group_by(Overall_Safety) %>%
      summarise(Count = n(), .groups = 'drop')
      
    color_map <- c("Safe" = "#2E86C1", "Dangerous" = "#DD4B39")
      
    plot_ly(safety_counts, 
            labels = ~Overall_Safety, 
            values = ~Count,
            type = "pie", 
            hole = 0.4,
            marker = list(colors = unname(color_map[safety_counts$Overall_Safety])),
            textinfo = "label+percent",
            hoverinfo = "label+percent+value") %>%
      layout(title = "Safety Status Distribution")
  })

  #Scatter Plot
  output$scatter_plot <- renderPlotly({
    p <- ggplot(filtered_df(), aes(x = CH4.emissions, y = N2O.emissions, 
                                    text = Country, color = Region)) +
      geom_point(size = 2, alpha = 0.7) +
      scale_x_log10() + 
      scale_y_log10() + 
      theme_minimal() +
      labs(title = "CH₄ vs N₂O Emissions (Log Scale)", 
           x = "CH₄ Emissions (Mt)", 
           y = "N₂O Emissions (Mt)")
    
    ggplotly(p, tooltip = c("text", "x", "y"))
  })
  
  #Regression Summary
  output$reg_summary <- renderPrint({
    req(nrow(filtered_df()) > 0)
    model <- lm(CH4.emissions ~ N2O.emissions, data = filtered_df())
    cat("LINEAR REGRESSION ANALYSIS\n")
    cat("==========================\n\n")
    cat("Model: CH₄ Emissions ~ N₂O Emissions\n\n")
    print(summary(model))
  })

  #Data Table
  output$data_table <- renderDT({
    datatable(
      filtered_df() %>% 
        select(Country, Region, CH4.emissions, N2O.emissions, Methane_Safety, ISO_Code),
      options = list(
        pageLength = 10, 
        scrollX = TRUE
      ),
      rownames = FALSE
    ) %>%
      formatRound(columns = c("CH4.emissions", "N2O.emissions"), digits = 2)
  })
}

shinyApp(ui, server)


Listening on http://127.0.0.1:6592

